# Numerical differentiation

```{admonition} Learning outcomes
After working through this chapter, you should be able to:

1. explain the difference between analytical and numerical differentiation
2. implement forward, backward and central differences
3. investigate how step size and floating-point rounding affect the result
4. differentiate experimental data and interpret the derivative chemically
```

The derivative can be written as $f'(x)$ or in Leibniz notation, $df/dx$. In chemistry the latter is especially useful because the variables carry physical meaning. For example, $dc/dt$ is a change in concentration with time, while $d\mathrm{pH}/dV$ is the change in pH with added volume.


The derivative is defined by a limit,

$$f'(x)=\lim_{\Delta x\rightarrow0}\frac{f(x+\Delta x)-f(x)}{\Delta x}.$$

A computer cannot use an infinitely small step. We therefore choose a small but finite $h$:

$$f'(x)\approx\frac{f(x+h)-f(x)}{h}.$$

This is the **forward difference**.


In [ ]:
def f(x):
    return 2*x + 2

x = 1.0
h = 1e-8
numerical_derivative = (f(x + h) - f(x)) / h
print("Numerical:", numerical_derivative)
print("Analytical:", 2.0)


## Forward, backward and central differences

$$f'(x)\approx\frac{f(x+h)-f(x)}{h}\qquad\text{forward}$$

$$f'(x)\approx\frac{f(x)-f(x-h)}{h}\qquad\text{backward}$$

$$f'(x)\approx\frac{f(x+h)-f(x-h)}{2h}\qquad\text{central}$$

For smooth functions, the central difference is usually substantially more accurate for the same step size.


In [ ]:
def derivative_forward(f, x, h=1e-6):
    return (f(x + h) - f(x)) / h

def derivative_backward(f, x, h=1e-6):
    return (f(x) - f(x - h)) / h

def derivative_central(f, x, h=1e-6):
    return (f(x + h) - f(x - h)) / (2*h)


## Error analysis: a smaller step is not always better

Two error mechanisms compete. **Truncation error** generally decreases as $h$ becomes smaller, while **floating-point round-off** becomes important when two nearly equal numbers are subtracted and the result is divided by a very small number. There is therefore no universal optimal value of $h$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def g(x):
    return np.sin(x)

x = 1.0
exact = np.cos(x)
step_sizes = np.logspace(-16, -1, 100)
forward_errors = [abs(derivative_forward(g, x, h) - exact) for h in step_sizes]
central_errors = [abs(derivative_central(g, x, h) - exact) for h in step_sizes]

plt.loglog(step_sizes, forward_errors, label="Forward")
plt.loglog(step_sizes, central_errors, label="Central")
plt.xlabel("Step size h")
plt.ylabel("Absolute error")
plt.legend()
plt.show()


## Differentiating experimental data

Experimental data are already discrete, so derivatives are estimated from neighbouring measurements. NumPy's `gradient` function is convenient because it can handle the spacing between measurement points. A derivative also amplifies noise, which is important when interpreting experimental chemical data.


In [ ]:
time = np.array([0, 10, 20, 30, 40, 50], dtype=float)
concentration = np.array([1.00, 0.82, 0.67, 0.55, 0.45, 0.37])
dc_dt = np.gradient(concentration, time)

for t, c, rate in zip(time, concentration, dc_dt):
    print(f"{t:4.0f} s   c = {c:.3f} mol/L   dc/dt = {rate:.5f} mol/(L s)")


```{admonition} Chemical interpretation
:class: note
For disappearance of a reactant A, $d[A]/dt$ is negative. The reaction rate is often defined with a minus sign, $v=-d[A]/dt$, so that the reported rate is positive.
```

## Exercises

1. Compare the three difference formulas for $f(x)=e^{-x}$ at $x=1$.
2. Make a log-log plot of the error as a function of $h$.
3. Add small random noise to a concentration-time data set and examine what happens to the derivative.
4. For a titration curve, explain why a maximum in $d\mathrm{pH}/dV$ can be useful.
